# Context Engineering -- A Practical Crash Course

**Five techniques for managing what an LLM sees, so it produces better outputs.**

This notebook consolidates five context engineering patterns into a single,
runnable reference. Every closed-source LLM call uses **Gemini 2.0 Flash**.
HuggingFace models (SentenceTransformer, Provence) are kept as-is for
Colab-friendly execution.

---

| # | Technique | Core Idea |
|---|-----------|-----------|
| 1 | **Tool Loadout** | Show the LLM only the tools it actually needs (Less-is-More) |
| 2 | **Context Quarantine** | Isolate each sub-task in its own context thread |
| 3 | **Context Pruning** | Remove irrelevant sentences before generation |
| 4 | **Context Summarization** | Condense growing context to preserve focus |
| 5 | **Context Offloading** | Give the agent a scratchpad / think tool outside the main context |

References:
- Drew Breunig -- How to Fix Your Context
- Anthropic -- Building a Multi-Agent Research System
- Provence: Efficient and Robust Context Pruning (ICLR 2025)
- Less is More: Optimizing Function Calling (arXiv 2411.15399)
- Anthropic -- Claude Think Tool

In [ ]:
# -- Setup and Imports --
# On Colab, uncomment the pip install line below.
# !pip install -q langchain langchain-google-genai sentence-transformers scikit-learn nltk python-dotenv

import os, json, time, textwrap
import numpy as np
import nltk
from typing import List, Dict, Any, Tuple
from concurrent.futures import ThreadPoolExecutor
from collections import deque

from dotenv import load_dotenv
from IPython.display import display, Markdown

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') or input('Enter Gemini API key: ').strip()

llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',
    google_api_key=GEMINI_API_KEY,
    temperature=0.7,
)

def get_llm(temperature=0.7):
    return ChatGoogleGenerativeAI(
        model='gemini-2.0-flash',
        google_api_key=GEMINI_API_KEY,
        temperature=temperature,
    )

def call_gemini(prompt: str, system: str = '', temperature: float = 0.7) -> str:
    msgs = []
    if system:
        msgs.append(SystemMessage(content=system))
    msgs.append(HumanMessage(content=prompt))
    return get_llm(temperature).invoke(msgs).content.strip()

resp = llm.invoke('Reply with exactly: CONTEXT ENGINEERING READY')
print(f'[OK] {resp.content}')

---
# Part 1 -- Tool Loadout (Less-is-More)

**Problem:** When we give an LLM access to many tools at once, it gets confused --
picking the wrong tool, hallucinating parameters, or slowing down.

**Solution (from arXiv 2411.15399):**
1. Ask the LLM to *describe* what tools it needs -- without showing any tools.
2. Use semantic similarity (SentenceTransformer, a HuggingFace model) to find the closest real tools.
3. Provide only those selected tools for the actual call.

This reduces confusion, improves accuracy, and cuts execution time.

```
BASELINE (all tools)                LESS-IS-MORE (selected tools)
+--------------------------+        +----------+  similarity  +----------+
| LLM sees 10 tools       |        | LLM says |  -------->   | top-3    |
| --> confused, slow       |        | "I need  |              | matched  |
+--------------------------+        | X and Y" |              | tools    |
                                    +----------+              +----------+
```

In [ ]:
# -- Part 1: Tool Loadout --
# We define a set of mock tools, then implement the Less-is-More selector.

from langchain.tools import tool as lc_tool

@lc_tool
def get_weather(location: str) -> str:
    """Get the current weather for a specific location."""
    return f"Weather in {location}: Sunny, 72F"

@lc_tool
def search_web(query: str) -> str:
    """Search the web for information about a topic."""
    return f"Search results for '{query}': Found 10 relevant articles"

@lc_tool
def calculate_math(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return f"Result: {eval(expression)}"
    except Exception:
        return "Error: Invalid expression"

@lc_tool
def translate_text(text: str, target_language: str) -> str:
    """Translate text to a target language."""
    return f"Translated '{text}' to {target_language}"

@lc_tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {recipient} with subject '{subject}'"

@lc_tool
def get_stock_price(symbol: str) -> str:
    """Get the current stock price for a given symbol."""
    return f"Stock price for {symbol}: $150.25"

@lc_tool
def create_calendar_event(title: str, date: str, time_str: str) -> str:
    """Create a calendar event with title, date, and time."""
    return f"Calendar event '{title}' created for {date} at {time_str}"

@lc_tool
def get_news(category: str) -> str:
    """Get latest news for a specific category."""
    return f"Latest {category} news: 5 articles found"

@lc_tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another."""
    return f"Converted {amount} {from_currency} to {to_currency}: {amount * 1.2}"

@lc_tool
def get_directions(origin: str, destination: str) -> str:
    """Get directions from origin to destination."""
    return f"Directions from {origin} to {destination}: 15 miles, 25 min"

ALL_TOOLS = [
    get_weather, search_web, calculate_math, translate_text, send_email,
    get_stock_price, create_calendar_event, get_news, convert_currency, get_directions,
]

print(f"[OK] Defined {len(ALL_TOOLS)} tools")


# -- Less-is-More Tool Selector --
# SentenceTransformer (HuggingFace) handles the embedding + similarity step.
# Gemini 2.0 Flash handles the tool-recommendation step.

class LessIsMoreToolSelector:
    def __init__(self, all_tools: list, top_k: int = 3):
        self.all_tools = all_tools
        self.top_k = top_k
        print("Loading SentenceTransformer (HuggingFace) for tool embeddings...")
        self.embed_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
        self.tool_descs = [f"{t.name}: {t.description}" for t in all_tools]
        self.tool_embeds = self.embed_model.encode(self.tool_descs)
        print(f"[OK] Embedded {len(all_tools)} tool descriptions")

    def recommend(self, query: str) -> List[str]:
        prompt = (
            f"Given the user query below, describe 1-3 tool descriptions that would "
            f"be needed. Return ONLY a JSON list of short descriptions.\n\n"
            f"User query: {query}\n\nTool descriptions (JSON list):"
        )
        raw = call_gemini(prompt, temperature=0)
        try:
            clean = raw.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
            return json.loads(clean)
        except Exception:
            return [raw]

    def select(self, query: str) -> Tuple[list, list]:
        recs = self.recommend(query)
        print(f"  LLM recommended: {recs}")
        rec_embeds = self.embed_model.encode(recs)
        sims = cosine_similarity(rec_embeds, self.tool_embeds)
        max_sims = sims.max(axis=0)
        top_idx = np.argsort(max_sims)[-self.top_k:][::-1]
        selected = [self.all_tools[i] for i in top_idx]
        return selected, top_idx.tolist()

    def run(self, query: str):
        print(f"\nQuery: {query}")
        print("-" * 50)
        selected, idx = self.select(query)
        print(f"  Selected {len(selected)} tools:")
        for t in selected:
            print(f"    - {t.name}: {t.description}")
        return selected

selector = LessIsMoreToolSelector(ALL_TOOLS, top_k=3)

In [ ]:
# -- Run examples --

selector.run("What's the weather like in San Francisco?")
selector.run("Check the stock price of AAPL and convert 1000 USD to EUR")
selector.run("Search for information about AI and email it to my colleague")

# -- Baseline vs Less-is-More comparison --
print("\n" + "=" * 60)
print("COMPARISON: Baseline (all tools) vs Less-is-More")
print("=" * 60)

test_q = "What is the weather in New York?"

# Baseline: show all 10 tools
tool_list = "\n".join(f"  {i+1}. {t.name}: {t.description}" for i, t in enumerate(ALL_TOOLS))
baseline_resp = call_gemini(
    f"We have these tools:\n{tool_list}\n\nQuery: {test_q}\nWhich tools would we use? List the tool names.",
    temperature=0,
)
print(f"\nBaseline (saw {len(ALL_TOOLS)} tools): {baseline_resp[:200]}")

# Less-is-More
selected, _ = selector.select(test_q)
print(f"Less-is-More (saw {len(selected)} tools): {[t.name for t in selected]}")
print(f"Reduction: {(1 - len(selected)/len(ALL_TOOLS))*100:.0f}% fewer tools shown to the LLM")

---
# Part 2 -- Context Quarantine

**Problem:** When a single agent handles multiple unrelated sub-tasks in one
conversation thread, its context accumulates noise from earlier tasks. This
causes *context confusion*, *context distraction*, and *context poisoning*
(errors propagate).

**Solution:** Give each sub-task its own agent with an isolated context. Then
synthesize the results.

Anthropic reported **90.2% improvement** over a single agent on research tasks
using this pattern, primarily because each sub-agent explores independently
without cross-contamination.

```
SINGLE AGENT (shared context)           QUARANTINED (isolated contexts)
+------------------------------+        +--------+  +--------+  +--------+
| Agent                        |        | Agent1 |  | Agent2 |  | Agent3 |
| Context: [T1, T2, T3, ...]  |        | T1     |  | T2     |  | T3     |
| --> confused, poisoned       |        +--------+  +--------+  +--------+
+------------------------------+              \        |        /
                                               v       v       v
                                            +---------------------+
                                            | Lead Agent: Combine |
                                            +---------------------+
```

In [ ]:
# -- Part 2: Context Quarantine --
# We build a simple agent class where each instance keeps its own message history.
# Then we compare single-agent (shared context) vs multi-agent (quarantined).

class SimpleAgent:
    """An agent with its own isolated context (conversation history)."""
    def __init__(self, system_prompt: str):
        self.system_prompt = system_prompt
        self.history: list = []

    def __call__(self, message: str) -> str:
        self.history.append(HumanMessage(content=message))
        msgs = [SystemMessage(content=self.system_prompt)] + self.history
        resp = llm.invoke(msgs)
        self.history.append(AIMessage(content=resp.content))
        return resp.content

    @property
    def context_size(self) -> int:
        return sum(len(m.content) for m in self.history)

# -- Problem: single agent, shared context --
print("WITHOUT QUARANTINE -- single agent, shared context")
print("=" * 55)

single = SimpleAgent("We are a helpful research assistant.")
topics = [
    "Research the serverless computing market size",
    "Identify top serverless providers",
    "Analyze serverless technical capabilities",
]
for t in topics:
    resp = single(t)
    print(f"  [{single.context_size:,} chars in context] {t}")
    print(f"    -> {resp[:120]}...\n")

print(f"[!] By the third task, the context contains {single.context_size:,} chars")
print("    of accumulated noise from unrelated sub-tasks.\n")

# -- Solution: quarantined agents --
print("WITH QUARANTINE -- separate agents, isolated contexts")
print("=" * 55)

agents = {
    "market": SimpleAgent("We are a market analyst. Focus ONLY on market size and growth."),
    "competitors": SimpleAgent("We are a competitive analyst. Focus ONLY on identifying key players."),
    "technical": SimpleAgent("We are a technology analyst. Focus ONLY on technical capabilities."),
}
results = {}
for name, agent in agents.items():
    resp = agent(topics[list(agents.keys()).index(name)])
    results[name] = resp
    print(f"  [{agent.context_size:,} chars] {name}: {resp[:120]}...\n")

print("[OK] Each agent has a clean, focused context. No cross-contamination.")

In [ ]:
# -- Orchestrator pattern: plan -> parallel subagents -> synthesize --

class ResearchOrchestrator:
    def research(self, query: str, aspects: List[str]) -> Dict:
        print(f"Query: {query}")
        print(f"Aspects: {aspects}")
        print("-" * 55)

        # Step 1: create quarantined subagents
        subagents = []
        for aspect in aspects:
            agent = SimpleAgent(f"We are a specialist in {aspect}. Focus ONLY on this aspect. Be concise.")
            subagents.append((aspect, agent))
            print(f"  [CREATED] subagent for: {aspect}")

        # Step 2: execute in parallel
        print("\n  [RUNNING] Subagents in parallel...")
        start = time.time()

        def run_one(pair):
            aspect, agent = pair
            return aspect, agent(f"Research {aspect} for: {query}")

        with ThreadPoolExecutor(max_workers=len(subagents)) as pool:
            findings = list(pool.map(run_one, subagents))

        elapsed = time.time() - start
        print(f"  [DONE] {len(findings)} subagents completed in {elapsed:.1f}s")

        # Step 3: synthesize with a lead agent
        print("  [SYNTHESIZING] Lead agent combining results...")
        synthesis_prompt = f"Synthesize these findings for: {query}\n\n"
        for aspect, text in findings:
            synthesis_prompt += f"**{aspect}**:\n{text[:300]}\n\n"
        synthesis_prompt += "Provide a brief integrated summary (5-8 sentences)."

        lead = SimpleAgent("We are a lead researcher who synthesizes findings concisely.")
        report = lead(synthesis_prompt)

        return {"findings": findings, "report": report, "elapsed": elapsed}


orch = ResearchOrchestrator()
result = orch.research("AI agent frameworks", [
    "market size and growth",
    "top frameworks and their features",
    "key technical architectures",
])

print("\n" + "=" * 55)
print("SYNTHESIZED REPORT")
print("=" * 55)
print(result["report"][:600])

---
# Part 3 -- Context Pruning

**Problem:** Long retrieved documents contain irrelevant sentences that waste
tokens and distract the model.

**Solution:** Remove irrelevant sentences *before* passing context to the LLM.

We demonstrate two approaches:
1. **LLM-based pruning** -- ask Gemini to identify relevant sentence indices.
2. **Provence** (HuggingFace, ICLR 2025) -- a DeBERTa-based model trained
   specifically for context pruning. Fast and accurate.

| Method | Speed | Quality | Cost |
|--------|-------|---------|------|
| No pruning | Fast | Baseline | High tokens |
| LLM pruning (Gemini) | Slow | High | Extra LLM call |
| Provence (HuggingFace) | Very fast | High | Model hosting only |

In [ ]:
# -- Part 3: Context Pruning --
# We start with a sample document that mixes relevant and irrelevant content.

SAMPLE_DOCUMENT = """
Amazon Web Services (AWS) is a subsidiary of Amazon that provides on-demand cloud computing platforms and APIs. AWS was launched in 2006 and has since become the most broadly adopted cloud platform.

The company headquarters are in Seattle, Washington. AWS employs over 100,000 people worldwide. The CEO of AWS is Matt Garman.

Amazon Bedrock is a fully managed service offering high-performing foundation models from AI21 Labs, Anthropic, Cohere, Meta, Mistral AI, Stability AI, and Amazon through a single API. Bedrock makes it easy to build and scale generative AI applications.

The AWS re:Invent conference is held annually in Las Vegas. It typically attracts over 50,000 attendees. The first re:Invent was held in 2012.

With Amazon Bedrock, we can experiment with and evaluate top FMs, customize them with fine-tuning and RAG, and build agents that execute tasks using enterprise data.

Bedrock supports Claude (Anthropic), Titan (Amazon), Llama (Meta), Command (Cohere), and Jurassic (AI21 Labs). The service is serverless -- no infrastructure to manage.

Amazon founder Jeff Bezos started the company in 1994 as an online bookstore.

Key Bedrock features include model customization, knowledge bases for RAG, agents for automation, guardrails, and model evaluation. It integrates with other AWS services.

The Amazon rainforest covers most of the Amazon basin of South America. This is unrelated to Amazon the company.
"""

print(f"Document length: {len(SAMPLE_DOCUMENT)} chars")
print(f"Sentences: {len(nltk.sent_tokenize(SAMPLE_DOCUMENT))}")


# -- Method 1: LLM-based pruning using Gemini --
def llm_prune_context(question: str, document: str) -> Dict:
    sentences = nltk.sent_tokenize(document)
    numbered = '\n'.join(f'[{i}] {s}' for i, s in enumerate(sentences))

    prompt = (
        f"Given the question and numbered sentences, return ONLY the sentence "
        f"numbers relevant to answering the question as a JSON array.\n\n"
        f"Question: {question}\n\nSentences:\n{numbered}\n\nRelevant indices:"
    )
    raw = call_gemini(prompt, temperature=0)
    try:
        clean = raw.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
        indices = json.loads(clean)
        pruned = ' '.join(sentences[i] for i in indices if i < len(sentences))
    except Exception:
        pruned = document
        indices = list(range(len(sentences)))

    return {
        'original_len': len(document),
        'pruned_len': len(pruned),
        'reduction': f"{(1 - len(pruned)/len(document))*100:.1f}%",
        'kept': len(indices),
        'total': len(sentences),
        'pruned_text': pruned,
        'indices': indices,
    }


question = "What is Amazon Bedrock and what models does it support?"
result = llm_prune_context(question, SAMPLE_DOCUMENT)

print(f"\nQuestion: {question}")
print(f"Original: {result['original_len']} chars, {result['total']} sentences")
print(f"Pruned:   {result['pruned_len']} chars, {result['kept']} sentences")
print(f"Reduction: {result['reduction']}")
print(f"\n--- Pruned Context ---\n{result['pruned_text']}")

In [ ]:
# -- Method 2: Provence model (HuggingFace, ICLR 2025) --
# Provence is a DeBERTa-based model trained for context pruning.
# It runs locally, no API calls needed. Fast and domain-agnostic.
# On Colab: !pip install -q transformers torch

from transformers import AutoModel

print("Loading Provence model (first run downloads ~1.75 GB)...")
provence = AutoModel.from_pretrained(
    "naver/provence-reranker-debertav3-v1",
    trust_remote_code=True,
)
print("[OK] Provence loaded")


def provence_prune(question: str, document: str) -> Dict:
    start = time.time()
    pruned_text = provence.process(question, document)
    # provence.process may return a string or a dict depending on version
    if isinstance(pruned_text, dict):
        pruned_text = pruned_text.get('pruned_context', str(pruned_text))
    elapsed = time.time() - start
    return {
        'original_len': len(document),
        'pruned_len': len(pruned_text),
        'reduction': f"{(1 - len(pruned_text)/len(document))*100:.1f}%",
        'pruned_text': pruned_text,
        'latency': f"{elapsed:.2f}s",
    }


question = "What is Amazon Bedrock and what models does it support?"

prov_result = provence_prune(question, SAMPLE_DOCUMENT)
print(f"Provence -- Reduction: {prov_result['reduction']}, Latency: {prov_result['latency']}")
print(f"Pruned text:\n{prov_result['pruned_text'][:400]}")


# -- Comparison: answer quality with full vs pruned context --
print("\n" + "=" * 60)
print("COMPARISON: Full Context vs LLM-Pruned vs Provence-Pruned")
print("=" * 60)

def answer_with_context(question: str, context: str) -> str:
    return call_gemini(
        f"Answer based ONLY on the context.\n\nContext:\n{context}\n\nQuestion: {question}",
        temperature=0,
    )

llm_result = llm_prune_context(question, SAMPLE_DOCUMENT)

for label, ctx in [
    ("Full context", SAMPLE_DOCUMENT),
    ("LLM-pruned",  llm_result['pruned_text']),
    ("Provence-pruned", prov_result['pruned_text']),
]:
    answer = answer_with_context(question, ctx)
    print(f"\n[{label}] ({len(ctx)} chars)")
    print(f"  {answer[:250]}...")

In [ ]:
# -- RAG Pipeline with Pruning --
# A complete retrieve -> prune -> generate pipeline.

CORPUS = [
    SAMPLE_DOCUMENT,
    """Amazon S3 is an object storage service with 11 nines of durability.
S3 classes include Standard, Intelligent-Tiering, Glacier, and Deep Archive.
The Great Wall of China is a fortification along China northern borders.""",
    """AWS Lambda is serverless compute. It supports Python, Node.js, Java, Go, Ruby.
Lambda can be triggered by API Gateway, S3, DynamoDB, SNS, CloudWatch.
Pizza is an Italian dish of leavened dough topped with tomatoes and cheese.""",
]


class RAGWithPruning:
    def __init__(self, documents: list, pruner=None):
        self.documents = documents
        self.pruner = pruner  # provence model or None

    def retrieve(self, query: str, top_k: int = 3) -> list:
        query_words = set(query.lower().split())
        scored = []
        for doc in self.documents:
            overlap = len(query_words & set(doc.lower().split()))
            scored.append((overlap, doc))
        scored.sort(reverse=True)
        return [doc for _, doc in scored[:top_k]]

    def prune(self, query: str, documents: list) -> str:
        pruned = []
        for doc in documents:
            if self.pruner:
                p = self.pruner.process(query, doc)
                text = p if isinstance(p, str) else p.get('pruned_context', str(p))
            else:
                text = doc
            if text.strip():
                pruned.append(text.strip())
        return '\n\n'.join(pruned)

    def query(self, question: str) -> Dict:
        start = time.time()
        retrieved = self.retrieve(question)
        context = self.prune(question, retrieved)
        answer = call_gemini(
            f"Context:\n{context}\n\nQuestion: {question}\nAnswer:",
            temperature=0,
        )
        return {
            'answer': answer,
            'context_len': len(context),
            'docs_retrieved': len(retrieved),
            'elapsed': f"{time.time()-start:.2f}s",
        }


rag = RAGWithPruning(CORPUS, pruner=provence)

q = "What foundation models are available in Amazon Bedrock?"
r = rag.query(q)
print(f"Question: {q}")
print(f"Retrieved {r['docs_retrieved']} docs, context {r['context_len']} chars, took {r['elapsed']}")
print(f"\nAnswer:\n{r['answer'][:400]}")

---
# Part 4 -- Context Summarization

**Problem:** As conversations or agent steps accumulate, the context grows
until the model starts repeating itself or losing focus (observed beyond
~100k tokens in practice).

**Solution:** Proactively summarize older context, preserving key facts,
decisions, and user preferences while discarding noise.

We implement three patterns:
1. **Conversation Summarizer** -- sliding window with auto-compression
2. **Hierarchical Summarization** -- summarize chunks, then summarize summaries
3. **Structured Context Manager** -- different sections get different treatment

In [ ]:
# -- Part 4: Context Summarization --

# -- 4A: Conversation Summarizer --
# Keeps the last N messages in full. When the buffer overflows,
# we summarize the older messages and inject the summary as system context.

class ConversationSummarizer:
    def __init__(self, system_prompt: str, max_messages: int = 10):
        self.system_prompt = system_prompt
        self.max_messages = max_messages
        self.messages: list = []
        self.summary: str | None = None

    def _summarize_history(self) -> str:
        history = '\n'.join(
            f"{m.type.upper()}: {m.content[:200]}" for m in self.messages
        )
        return call_gemini(
            f"Summarize this conversation, preserving key decisions, facts, "
            f"and preferences.\n\n{history}",
            temperature=0.3,
        )

    def _maybe_compress(self):
        if len(self.messages) >= self.max_messages:
            self.summary = self._summarize_history()
            self.messages = self.messages[-4:]
            print(f"  [COMPRESSED] Summary: {self.summary[:100]}...")

    def chat(self, user_input: str) -> str:
        self._maybe_compress()
        sys = self.system_prompt
        if self.summary:
            sys += f"\n\nPrevious conversation summary:\n{self.summary}"
        self.messages.append(HumanMessage(content=user_input))
        msgs = [SystemMessage(content=sys)] + self.messages
        resp = llm.invoke(msgs)
        self.messages.append(AIMessage(content=resp.content))
        return resp.content

    def stats(self):
        return {'messages': len(self.messages), 'has_summary': self.summary is not None}


chat_agent = ConversationSummarizer(
    "We are a helpful coding assistant.", max_messages=6
)

questions = [
    "What is the best way to handle errors in Python?",
    "Can we see a try-except example?",
    "What about custom exceptions?",
    "How should we log errors properly?",
    "What logging levels should we use?",
]

for q in questions:
    print(f"\n[USER] {q}")
    resp = chat_agent.chat(q)
    print(f"[AGENT] {resp[:150]}...")
    print(f"  Stats: {chat_agent.stats()}")

In [ ]:
# -- 4B: Hierarchical Summarization --
# For very long documents: summarize chunks first, then summarize the summaries.

def chunk_text(text: str, chunk_size: int = 2000) -> List[str]:
    words = text.split()
    return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

def hierarchical_summarize(text: str, focus: str = None) -> Dict:
    chunks = chunk_text(text)
    if len(chunks) == 1:
        return {'summary': call_gemini(f"Summarize briefly:\n{text}", temperature=0.3), 'chunks': 1}
    chunk_sums = [call_gemini(
        f"Summarize this chunk briefly{' focusing on '+focus if focus else ''}:\n{c}",
        temperature=0.3,
    ) for c in chunks]
    combined = '\n\n'.join(f"Section {i+1}: {s}" for i, s in enumerate(chunk_sums))
    final = call_gemini(f"Combine these section summaries into one:\n{combined}", temperature=0.3)
    return {'summary': final, 'chunks': len(chunks), 'chunk_summaries': chunk_sums}

# Quick demo
long_text = ("AWS provides compute (EC2, Lambda), storage (S3, EBS), databases (RDS, DynamoDB), "
             "networking (VPC, Route53), security (IAM, KMS), ML (SageMaker, Bedrock), "
             "and analytics (Athena, Redshift, Kinesis). ") * 15

r = hierarchical_summarize(long_text, focus="key AWS services")
print(f"Chunks: {r['chunks']}")
print(f"Summary: {r['summary'][:300]}")


# -- 4C: Structured Context Manager --
# Different context sections need different treatment.

class StructuredContextManager:
    def __init__(self):
        self.sections = {'goals': [], 'facts': [], 'history': [], 'scratchpad': []}
        self.summaries: dict = {}

    def add(self, section: str, content: str):
        if section in self.sections:
            self.sections[section].append(content)

    def _maybe_summarize(self, section: str, threshold: int):
        content = '\n'.join(self.sections[section])
        if len(content) > threshold:
            self.summaries[section] = call_gemini(
                f"Summarize concisely:\n{content}", temperature=0.3
            )
            self.sections[section] = []

    def compile(self) -> str:
        self._maybe_summarize('facts', 2000)
        self._maybe_summarize('history', 1000)
        parts = []
        if self.sections['goals']:
            parts.append("## Goals\n" + '\n'.join(self.sections['goals']))
        facts = self.summaries.get('facts') or '\n'.join(self.sections['facts'])
        if facts:
            parts.append(f"## Key Facts\n{facts}")
        history = self.summaries.get('history') or '\n'.join(self.sections['history'])
        if history:
            parts.append(f"## History\n{history}")
        return '\n\n'.join(parts)


ctx = StructuredContextManager()
ctx.add('goals', 'Build a REST API for user management')
ctx.add('goals', 'Use Python with FastAPI')
ctx.add('facts', 'Database: PostgreSQL on RDS')
ctx.add('facts', 'Auth: JWT tokens, 1-hour expiry')
ctx.add('history', 'Created project structure')
ctx.add('history', 'Implemented CRUD endpoints')

print("\n" + "=" * 55)
print("STRUCTURED CONTEXT")
print("=" * 55)
print(ctx.compile())


# -- 4D: Progressive Summarization Agent --
# Maintains a running summary that gets updated after each step.

class ProgressiveAgent:
    def __init__(self, task: str):
        self.task = task
        self.running_summary = f"Task: {task}"
        self.step_count = 0

    def step(self, description: str) -> str:
        self.step_count += 1
        result = call_gemini(
            f"Context:\n{self.running_summary}\n\nExecute this step: {description}",
        )
        self.running_summary = call_gemini(
            f"Current summary:\n{self.running_summary}\n\n"
            f"New step: {description}\nResult: {result[:300]}\n\n"
            f"Update the summary. Keep it under 200 words. Preserve the task, "
            f"key decisions, and current progress.",
            temperature=0.3,
        )
        return result


agent = ProgressiveAgent("Design a caching strategy for an e-commerce API")
for s in ["Identify what to cache", "Choose Redis vs Memcached", "Define invalidation strategy"]:
    print(f"\n[STEP] {s}")
    r = agent.step(s)
    print(f"  Result: {r[:150]}...")

print(f"\n[RUNNING SUMMARY after {agent.step_count} steps]")
print(agent.running_summary[:400])

---
# Part 5 -- Context Offloading

**Problem:** When the agent reasons, fetches tool outputs, and checks policies
all in the same message thread, the context gets noisy fast. Intermediate
reasoning clutter distracts the model from the actual task.

**Solution: the "think" tool** -- give the model a dedicated scratchpad that
does not pollute the main conversation context.

From Anthropic's research:
- **54% improvement** on airline customer service tasks
- **3.7% improvement** on retail tasks
- The think tool is especially powerful in **policy-heavy environments** where
  the agent must verify compliance before acting.

We implement this using LangChain tool calling with Gemini 2.0 Flash.

In [ ]:
# -- Part 5: Context Offloading --
# We define a "think" tool that the agent can call to reason without
# adding noise to the main conversation.

@tool
def think(thought: str) -> str:
    """Use this tool to think through a problem step by step. It does not
    obtain new information or change any state -- it just records our
    reasoning for later reference."""
    return "Thought recorded."

# We also define a couple of domain tools for a richer demo.
@tool
def get_order(order_id: str) -> str:
    """Look up an order by its ID and return details."""
    orders = {
        "ORD-123": json.dumps({
            "status": "delivered", "total": 150.00,
            "items": ["Widget A", "Widget B"],
            "delivery_date": "2025-11-25", "customer_tier": "gold",
        }),
        "ORD-456": json.dumps({
            "status": "shipped", "total": 89.99,
            "items": ["Gadget X"],
            "delivery_date": None, "customer_tier": "standard",
        }),
    }
    return orders.get(order_id, '{"error": "Order not found"}')

@tool
def process_refund(order_id: str, amount: float, reason: str) -> str:
    """Process a refund for a given order."""
    return json.dumps({"status": "refund_processed", "confirmation": f"REF-{order_id}"})


class ThinkingAgent:
    """Agent that can use a 'think' tool to reason before acting."""

    def __init__(self, system_prompt: str, tools: list):
        self.thought_log: list = []
        self.tool_map = {t.name: t for t in tools}
        self.llm_with_tools = get_llm(0.3).bind_tools(tools)
        self.system_prompt = system_prompt

    def run(self, user_input: str, max_turns: int = 10) -> str:
        messages = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=user_input),
        ]
        for _ in range(max_turns):
            resp = self.llm_with_tools.invoke(messages)
            messages.append(resp)
            if not resp.tool_calls:
                return resp.content
            for tc in resp.tool_calls:
                name, args, tid = tc['name'], tc['args'], tc['id']
                if name == 'think':
                    self.thought_log.append(args.get('thought', ''))
                    print(f"  [THINK] {args.get('thought', '')[:200]}...")
                else:
                    print(f"  [TOOL]  {name}({args})")
                result = self.tool_map[name].invoke(args)
                messages.append(ToolMessage(content=str(result), tool_call_id=tid))
        return "Agent reached max turns."


# -- Demo: simple problem solving --
simple_agent = ThinkingAgent(
    system_prompt=(
        "We are a helpful assistant that thinks carefully before responding. "
        "Before answering complex questions, use the think tool to break down "
        "the problem and consider different approaches."
    ),
    tools=[think],
)

print("THINK TOOL DEMO -- problem solving")
print("=" * 55)
answer = simple_agent.run(
    "Should we use a monolith or microservices for a new e-commerce platform "
    "that needs to handle Black Friday traffic spikes?"
)
print(f"\n[ANSWER]\n{answer[:500]}")
print(f"\n[THOUGHT LOG] ({len(simple_agent.thought_log)} thoughts recorded)")
for i, t in enumerate(simple_agent.thought_log):
    print(f"  {i+1}. {t[:200]}")

In [ ]:
# -- Policy-heavy environment --
# The think tool shines when the agent must check rules before acting.

REFUND_POLICY_PROMPT = """We are a customer service agent. Before processing ANY
refund, we MUST use the think tool to verify policy compliance.

REFUND POLICY:
- Full refund within 30 days of delivery
- 50% refund between 30-60 days
- No refund after 60 days
- Gold/Platinum customers get extended 90-day full refund window
- Damaged items always get full refund regardless of time

Steps before refunding:
1. Use think tool to list which policy rules apply
2. Check the order details against each rule
3. Calculate the correct refund amount
4. Verify the decision is compliant
5. Only then call process_refund
"""

policy_agent = ThinkingAgent(
    system_prompt=REFUND_POLICY_PROMPT,
    tools=[think, get_order, process_refund],
)

print("POLICY AGENT DEMO -- refund request")
print("=" * 55)
response = policy_agent.run(
    "We would like a refund for order ORD-123. The product was not what we expected."
)
print(f"\n[AGENT RESPONSE]\n{response[:500]}")
print(f"\n[THOUGHT LOG] ({len(policy_agent.thought_log)} thoughts)")
for i, t in enumerate(policy_agent.thought_log):
    print(f"  {i+1}. {t[:250]}")


# -- Extended scratchpad with categories --
@tool
def note(category: str, content: str) -> str:
    """Save a note to our scratchpad. category must be one of: facts, decisions, questions, todos."""
    return f"Noted under {category}."

@tool
def recall(category: str) -> str:
    """Recall all notes from a scratchpad category. Use 'all' for everything."""
    return "Recalled."


print("\n" + "=" * 55)
print("EXTENDED SCRATCHPAD DEMO")
print("=" * 55)

architect = ThinkingAgent(
    system_prompt=(
        "We are a technical architect. Use the note tool to organize our thinking: "
        "note(facts, ...) for constraints, note(decisions, ...) for choices, "
        "note(questions, ...) for open items, note(todos, ...) for action items. "
        "Use think for deeper reasoning."
    ),
    tools=[think, note, recall],
)

resp = architect.run(
    "Help us design a notification system for a mobile app. Requirements: "
    "push notifications, in-app notifications, email digests. "
    "We have 1M daily active users and need 10M notifications per day."
)
print(f"\n[RECOMMENDATION]\n{resp[:500]}")

In [ ]:
# -- Comparison: with vs without think tool --
# We send the same complex question to two setups and compare.

question = (
    "A customer wants to return order ORD-123 bought 40 days ago. "
    "They are a gold-tier customer. The item is not damaged. "
    "What refund should they get according to our policy?"
)

# WITHOUT think tool -- direct answer
print("WITHOUT THINK TOOL")
print("-" * 55)
direct_answer = call_gemini(
    f"{REFUND_POLICY_PROMPT}\n\n"
    f"Order ORD-123: delivered 40 days ago, total $150, gold customer, not damaged.\n\n"
    f"Question: {question}",
    temperature=0,
)
print(direct_answer[:400])

# WITH think tool -- structured reasoning
print("\nWITH THINK TOOL")
print("-" * 55)
think_agent = ThinkingAgent(
    system_prompt=REFUND_POLICY_PROMPT,
    tools=[think, get_order, process_refund],
)
think_answer = think_agent.run(question)
print(f"Answer: {think_answer[:400]}")
print(f"\nThoughts recorded: {len(think_agent.thought_log)}")
for i, t in enumerate(think_agent.thought_log):
    print(f"  {i+1}. {t[:200]}")

print("\n" + "=" * 55)
print("TAKEAWAY")
print("=" * 55)
print("The think-tool version explicitly reasons through each policy rule")
print("before deciding. This is auditable and less prone to errors.")
print("The direct version may skip steps or misapply rules silently.")

---
# Summary

| Technique | What It Does | When to Use |
|-----------|-------------|-------------|
| **Tool Loadout** | Dynamically select relevant tools via embeddings | Many tools available; model picks wrong ones |
| **Context Quarantine** | Isolate sub-tasks in separate agent contexts | Multi-aspect research; parallel work |
| **Context Pruning** | Remove irrelevant sentences before generation | RAG pipelines; long documents |
| **Context Summarization** | Compress growing context into summaries | Long conversations; multi-step agents |
| **Context Offloading** | Give the agent a think/scratchpad tool | Policy-heavy decisions; complex reasoning |

### Key Insight

> Context engineering is not about stuffing more into the prompt.
> It is about **controlling what the model sees** so it can focus on what matters.

All five techniques share the same principle: **less noise, better signal**.
We reduce, isolate, compress, or offload context so the LLM works with a
clean, focused window -- and produces higher-quality outputs as a result.